# M2 CLV-conditioned modulation 체크포인트 진단

기존 Dunnhumby seed 42 체크포인트를 재학습하지 않고 `none`, `N-only`, `V-only`, `both`, `shuffled-user`로 평가합니다. validation 채점만 수행하며 test/holdout은 사용하지 않습니다.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

REVIEWED_SHA = 'b51108b09a87705dbb7e048253530fdb06db13d6'
!rm -rf /content/clv-m2-lightgcn-runner
!git clone -q https://github.com/jung-un/clv-m2-lightgcn-runner.git /content/clv-m2-lightgcn-runner
%cd /content/clv-m2-lightgcn-runner
!git checkout -q {REVIEWED_SHA}
import subprocess
assert subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip() == REVIEWED_SHA


In [ ]:
import torch
from lightgcn_clv_modulation import configure_modulation_dunnhumby_run
from lightgcn_clv_modulation_diagnostic import run_checkpoint_diagnostics

assert torch.cuda.is_available(), '런타임 유형에서 GPU를 선택하세요.'
OUT_DIR = '/content/drive/MyDrive/논문/data/results_v3_dunnhumby_m2_clv_conditioned_modulation_v1'
cfg = configure_modulation_dunnhumby_run(out_dir=OUT_DIR)
diagnostic_df = run_checkpoint_diagnostics(cfg)


In [ ]:
from IPython.display import display

columns = [
    'view', 'recall@10', 'ndcg@10', 'recall@20', 'ndcg@20',
    'recall@50', 'ndcg@50', 'revenue@10', 'revenue@20', 'revenue@50',
    'arp@10', 'coverage@10', 'n_distinct@10', 'eff_catalog@10',
    'top10_share@10', 'top100_share@10', 'value_alignment',
]
display(diagnostic_df[[c for c in columns if c in diagnostic_df.columns]])
print('none 대비 paired delta:')
display(diagnostic_df.attrs['paired'])
print('modulation 구조 진단:')
display(diagnostic_df.attrs['structure'])
print('결과 파일:', diagnostic_df.attrs['paths'])
